In [ ]:
import tensorflow as tf
import numpy as np
import time
import os
import glob
from PIL import Image

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN & THÔNG SỐ
# ==========================================
BASE_PATH = "C:/DUT/Ki_2/Edge_AI/dataset/"
TRAIN_DIR = os.path.join(BASE_PATH, "train")

IMG_HEIGHT = 32
IMG_WIDTH = 32
TFLITE_MODEL_PATH = "traffic_sign_model_quantized.tflite"

print("Đang tải mô hình TFLite...")
# Khởi tạo TFLite Interpreter
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_scale, input_zero_point = input_details[0]['quantization']

# ==========================================
# HÀM LẤY DỮ LIỆU VALIDATION THỦ CÔNG ĐỂ TEST
# ==========================================
# Lấy 20% dữ liệu từ mỗi class trong thư mục train để làm tập test nội bộ
val_images = []
val_labels = []

classes = sorted(os.listdir(TRAIN_DIR))
for class_id, class_name in enumerate(classes):
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if os.path.isdir(class_dir):
        images_in_class = glob.glob(os.path.join(class_dir, '*.*'))
        # Lấy 20% ảnh cuối cùng của mỗi class làm validation
        val_split_idx = int(len(images_in_class) * 0.8)
        
        for img_path in images_in_class[val_split_idx:]:
            val_images.append(img_path)
            val_labels.append(class_id)

print(f"Tổng số ảnh Validation để test: {len(val_images)}")

# ==========================================
# ĐÁNH GIÁ ACCURACY VÀ INFERENCE TIME
# ==========================================
correct_predictions = 0
total_inference_time = 0

print("\nBắt đầu chạy suy luận (Inference)...")

for i in range(len(val_images)):
    img_path = val_images[i]
    true_label = val_labels[i]
    
    # 1. Tiền xử lý ảnh
    img = Image.open(img_path).convert('RGB').resize((IMG_WIDTH, IMG_HEIGHT))
    img_array = np.array(img, dtype=np.float32) / 255.0 # Chuẩn hóa [0, 1]
    
    # Ép kiểu về INT8 theo thông số lượng tử hóa của mô hình
    if input_scale != 0:
        img_array = img_array / input_scale + input_zero_point
    img_array = np.expand_dims(img_array.astype(np.int8), axis=0)
    
    # Nạp dữ liệu vào mô hình
    interpreter.set_tensor(input_details[0]['index'], img_array)
    
    # 2. Bắt đầu đo thời gian Inference
    start_time = time.time()
    interpreter.invoke()
    end_time = time.time()
    
    # 3. Lấy kết quả và cộng dồn thời gian
    total_inference_time += (end_time - start_time)
    
    output_data = interpreter.get_tensor(output_details[0]['index'])[0]
    predicted_class = np.argmax(output_data)
    
    if predicted_class == true_label:
        correct_predictions += 1

# ==========================================
# XUẤT BÁO CÁO (LOG)
# ==========================================
accuracy = (correct_predictions / len(val_images)) * 100
avg_inference_time_ms = (total_inference_time / len(val_images)) * 1000

print("\n" + "="*40)
print("📊 BÁO CÁO ĐÁNH GIÁ MÔ HÌNH TFLITE (INT8)")
print("="*40)
print(f"🎯 Độ chính xác (Accuracy): {accuracy:.2f}%")
print(f"⏱️ Tổng thời gian chạy:     {total_inference_time*1000:.4f}ms")
print(f"⚡ Tốc độ trung bình:       {avg_inference_time_ms:.2f} ms / ảnh")
print("="*40)

Đang tải mô hình TFLite...
Tổng số ảnh Validation để test: 1400

Bắt đầu chạy suy luận (Inference)...


c:\Users\ngong\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



📊 BÁO CÁO ĐÁNH GIÁ MÔ HÌNH TFLITE (INT8)
🎯 Độ chính xác (Accuracy): 99.50%
⏱️ Tổng thời gian chạy:     86.5891ms
⚡ Tốc độ trung bình:       0.06 ms / ảnh
